# Task
Train a DistilBERT model for token labeling on a custom resume dataset.

## Define `num classes`

### Subtask:
Determine and define the number of unique token labels in your dataset.


In [ ]:
!pip install -U transformers==4.46.0 accelerate evaluate seqeval

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification
import evaluate
import numpy as np


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "valid.jsonl"
    }
)

labels = [
    "O", "B-NAME", "I-NAME", "B-AGE", "I-AGE",
    "B-GENDER", "I-GENDER", "B-ADDRESS", "I-ADDRESS",
    "B-EMAIL", "I-EMAIL", "B-PHONE", "I-PHONE",
    "B-EDUCATION", "I-EDUCATION", "B-SKILL", "I-SKILL",
    "B-EXPERIENCE", "I-EXPERIENCE",
]

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
num_labels = len(labels)



LOAD DISTILBERT BASE MODEL

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at dslim/bert-base-NER and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([19]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768])

TOKENIZING

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    new_labels = []
    for i, labels_per_example in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens ([CLS], [SEP], etc.)
            if word_idx is None:
                label_ids.append(-100)
                continue

            # If tokenizer added subwords that don’t have labels
            if word_idx >= len(labels_per_example):
                label_ids.append(-100)
                continue

            # Label the first subword of each word, ignore others
            if word_idx != previous_word_idx:
                label_ids.append(label2id.get(labels_per_example[word_idx], 0))
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        new_labels.append(label_ids)

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs


In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

CHECKERS

In [ ]:
print(tokenized_dataset)
print(tokenized_dataset["train"].column_names)


DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['tokens', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 800
    })
})
['tokens', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']


TRAINING SETUP

In [ ]:
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

data_collator = DataCollatorForTokenClassification(tokenizer)

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p_i] for (p_i, l_i) in zip(pred, lab) if l_i != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l_i] for (p_i, l_i) in zip(pred, lab) if l_i != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }


TRAINING CONFIGS

In [ ]:
training_args = TrainingArguments(
    output_dir="./resume-ner",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.000600,0.000226,1.000000,1.000000,1.000000,1.000000
2,0.000300,0.000237,0.999449,0.999724,0.999586,0.999974
3,0.000200,0.000083,1.000000,1.000000,1.000000,1.000000
4,0.000200,0.000067,1.000000,1.000000,1.000000,1.000000
5,0.000200,0.000061,1.000000,1.000000,1.000000,1.000000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument i

TrainOutput(global_step=4000, training_loss=0.004312833994132233, metrics={'train_runtime': 20542.4611, 'train_samples_per_second': 0.779, 'train_steps_per_second': 0.195, 'total_flos': 811461335733480.0, 'train_loss': 0.004312833994132233, 'epoch': 5.0})

SAVE MODEL

In [ ]:
trainer.save_model("./resume-ner-model")
tokenizer.save_pretrained("./resume-ner-model")


('./resume-ner-model/tokenizer_config.json',
 './resume-ner-model/special_tokens_map.json',
 './resume-ner-model/vocab.txt',
 './resume-ner-model/added_tokens.json',
 './resume-ner-model/tokenizer.json')

TEST MODEL

In [ ]:
from transformers import pipeline

resume_ner = pipeline(
    "token-classification",
    model="./resume-ner-model",
    tokenizer="./resume-ner-model",
    aggregation_strategy="simple"
)

text = """
Dela Cruz, Zanille John is a 22-year-old male studying at the University of the Philippines.
He has experience in Python, Java, and ReactJS, and lives in Quezon City.
You can reach him at johndoe@gmail.com or +63 912 345 6789.
"""

results = resume_ner(text)
for r in results:
    print(r)

{'entity_group': 'NAME', 'score': np.float32(0.9998647), 'word': 'Del', 'start': 1, 'end': 4}
{'entity_group': 'NAME', 'score': np.float32(0.8594419), 'word': '##a Cruz', 'start': 4, 'end': 10}
{'entity_group': 'NAME', 'score': np.float32(0.9995338), 'word': 'Z', 'start': 12, 'end': 13}
{'entity_group': 'NAME', 'score': np.float32(0.662551), 'word': '##ani', 'start': 13, 'end': 16}
{'entity_group': 'NAME', 'score': np.float32(0.99440104), 'word': '##lle John', 'start': 16, 'end': 24}
{'entity_group': 'GENDER', 'score': np.float32(0.9938432), 'word': 'male', 'start': 42, 'end': 46}
{'entity_group': 'SKILL', 'score': np.float32(0.99981374), 'word': 'Python', 'start': 115, 'end': 121}
{'entity_group': 'SKILL', 'score': np.float32(0.9998036), 'word': 'Java', 'start': 123, 'end': 127}
{'entity_group': 'SKILL', 'score': np.float32(0.99980956), 'word': 'Re', 'start': 133, 'end': 135}
{'entity_group': 'SKILL', 'score': np.float32(0.99940634), 'word': '##actJS', 'start': 135, 'end': 140}
{'enti

SAVE AS ZIP

In [ ]:
!zip -r NER_Resume.zip resume-ner-model/

  adding: resume-ner-model/ (stored 0%)
  adding: resume-ner-model/vocab.txt (deflated 49%)
  adding: resume-ner-model/tokenizer_config.json (deflated 76%)
  adding: resume-ner-model/training_args.bin (deflated 53%)
  adding: resume-ner-model/model.safetensors (deflated 7%)
  adding: resume-ner-model/config.json (deflated 59%)
  adding: resume-ner-model/tokenizer.json (deflated 70%)
  adding: resume-ner-model/special_tokens_map.json (deflated 42%)
